# 07 — Baseline Comparison

Compare the Vision Transformer against baseline architectures as described
in the paper:

| Model | Type | Description |
|-------|------|-------------|
| Random Forest | ML | Trained on mean-pooled 64-dim embeddings |
| XGBoost | ML | Trained on mean-pooled 64-dim embeddings |
| FCN (Classifier) | DL | Fully-connected on pooled embeddings |
| CNN | DL | Spatial convolutions on embedding tiles |
| **ViT (ours)** | DL | Self-attention on patch embeddings |

**Inputs:**
- `.npz` embedding files in `output/dataset/`

**Outputs:**
- Accuracy, macro-F1, and confusion matrices for each model

In [ ]:
import numpy as np
import torch
import glob
import os
import re
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

EMBEDDING_DIR = '../../output/dataset/'
CATEGORY_NAMES = ['1-20%', '21-40%', '41-60%', '61-80%', '81-100%']

## 1. Load All Embeddings

In [ ]:
def parse_label(filename):
    """Extract category (0-indexed label) from filename like cat2_id50.0_cov0.30.npz."""
    m = re.match(r'cat(\d+)_id[\d.]+_cov[\d.]+\.npz', filename)
    if m:
        return int(m.group(1)) - 1  # 0-indexed
    return None


npz_files = sorted(glob.glob(os.path.join(EMBEDDING_DIR, '*.npz')))
print(f'Found {len(npz_files)} files.')

embeddings_list = []
labels_list = []

for fpath in tqdm(npz_files, desc='Loading'):
    label = parse_label(Path(fpath).name)
    if label is None:
        continue
    data = np.load(fpath)
    key = 'embeddings' if 'embeddings' in data else list(data.keys())[0]
    emb = data[key].astype(np.float32)  # (64, H, W)
    embeddings_list.append(emb)
    labels_list.append(label)

labels = np.array(labels_list)
print(f'Loaded {len(labels)} samples. Class distribution:')
for c, name in enumerate(CATEGORY_NAMES):
    print(f'  {name}: {(labels == c).sum()}')

## 2. Prepare Features for ML Baselines

For Random Forest and XGBoost, we compute **spatial mean** and **std** over each
of the 64 bands, giving a 128-dimensional feature vector per tile.

In [ ]:
# Compute mean + std over spatial dims for each band → (N, 128)
X_ml = np.array([
    np.concatenate([emb.mean(axis=(1, 2)), emb.std(axis=(1, 2))])
    for emb in tqdm(embeddings_list, desc='Extracting features')
])

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_ml, labels, np.arange(len(labels)),
    test_size=0.2, random_state=42, stratify=labels
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## 3. Random Forest Baseline

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_preds)
rf_f1 = f1_score(y_test, rf_preds, average='macro')
print(f'Random Forest — Accuracy: {rf_acc:.4f}  |  Macro-F1: {rf_f1:.4f}')

## 4. XGBoost Baseline

In [ ]:
try:
    from xgboost import XGBClassifier

    xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                        use_label_encoder=False, eval_metric='mlogloss',
                        random_state=42, n_jobs=-1)
    xgb.fit(X_train, y_train)
    xgb_preds = xgb.predict(X_test)

    xgb_acc = accuracy_score(y_test, xgb_preds)
    xgb_f1 = f1_score(y_test, xgb_preds, average='macro')
    print(f'XGBoost — Accuracy: {xgb_acc:.4f}  |  Macro-F1: {xgb_f1:.4f}')
except ImportError:
    print('XGBoost not installed. pip install xgboost to enable this baseline.')
    xgb_acc = xgb_f1 = None

## 5. Deep Learning Baselines (FCN + CNN + ViT)

Quick training loop for the three deep learning architectures.

In [ ]:
from model.model_architectures import Classifier, CNNClassifier, ViTClassifier

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES = len(CATEGORY_NAMES)
EPOCHS = 20
BATCH_SIZE = 8
LR = 1e-3


def train_and_evaluate(model, train_embs, train_labels, test_embs, test_labels,
                       epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR):
    """Simple training loop, returns (accuracy, macro-f1) on test set."""
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    # Create simple batches
    n = len(train_embs)
    for epoch in range(epochs):
        model.train()
        perm = np.random.permutation(n)
        total_loss = 0
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            x = torch.stack([torch.from_numpy(train_embs[i]) for i in idx]).to(DEVICE)
            y = torch.tensor([train_labels[i] for i in idx], dtype=torch.long).to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    # Evaluate
    model.eval()
    all_preds = []
    with torch.no_grad():
        for start in range(0, len(test_embs), batch_size):
            x = torch.stack([
                torch.from_numpy(test_embs[i])
                for i in range(start, min(start + batch_size, len(test_embs)))
            ]).to(DEVICE)
            logits = model(x)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())

    acc = accuracy_score(test_labels, all_preds)
    f1 = f1_score(test_labels, all_preds, average='macro')
    return acc, f1, np.array(all_preds)


# Prepare numpy arrays for train/test by index
train_embs = [embeddings_list[i] for i in idx_train]
test_embs = [embeddings_list[i] for i in idx_test]
train_labels_dl = y_train
test_labels_dl = y_test

In [ ]:
# FCN Baseline
print('Training FCN (Classifier)...')
fcn = Classifier(in_channels=64, out_features=NUM_CLASSES)
fcn_acc, fcn_f1, fcn_preds = train_and_evaluate(
    fcn, train_embs, train_labels_dl, test_embs, test_labels_dl)
print(f'FCN — Accuracy: {fcn_acc:.4f}  |  Macro-F1: {fcn_f1:.4f}')

In [ ]:
# CNN Baseline
print('Training CNN...')
cnn = CNNClassifier(in_channels=64, out_features=NUM_CLASSES)
cnn_acc, cnn_f1, cnn_preds = train_and_evaluate(
    cnn, train_embs, train_labels_dl, test_embs, test_labels_dl)
print(f'CNN — Accuracy: {cnn_acc:.4f}  |  Macro-F1: {cnn_f1:.4f}')

In [ ]:
# ViT (ours)
print('Training ViT...')
H, W = embeddings_list[0].shape[1], embeddings_list[0].shape[2]
vit = ViTClassifier(in_channels=64, out_features=NUM_CLASSES,
                     img_size=H, patch_size=16, embed_dim=128,
                     depth=4, num_heads=4)
vit_acc, vit_f1, vit_preds = train_and_evaluate(
    vit, train_embs, train_labels_dl, test_embs, test_labels_dl, epochs=30)
print(f'ViT — Accuracy: {vit_acc:.4f}  |  Macro-F1: {vit_f1:.4f}')

## 6. Summary Table

In [ ]:
import pandas as pd

rows = [
    {'Model': 'Random Forest', 'Type': 'ML', 'Accuracy': rf_acc, 'Macro-F1': rf_f1},
    {'Model': 'FCN', 'Type': 'DL', 'Accuracy': fcn_acc, 'Macro-F1': fcn_f1},
    {'Model': 'CNN', 'Type': 'DL', 'Accuracy': cnn_acc, 'Macro-F1': cnn_f1},
    {'Model': 'ViT (ours)', 'Type': 'DL', 'Accuracy': vit_acc, 'Macro-F1': vit_f1},
]
if xgb_acc is not None:
    rows.insert(1, {'Model': 'XGBoost', 'Type': 'ML', 'Accuracy': xgb_acc, 'Macro-F1': xgb_f1})

summary = pd.DataFrame(rows)
summary = summary.sort_values('Macro-F1', ascending=False).reset_index(drop=True)
print(summary.to_string(index=False))

## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, preds, name in zip(axes,
                            [rf_preds, cnn_preds, vit_preds],
                            ['Random Forest', 'CNN', 'ViT (ours)']):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=CATEGORY_NAMES)
    disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    ax.set_title(name)

plt.suptitle('Confusion Matrices — Baseline Comparison', fontsize=14)
plt.tight_layout()
plt.show()